# 확산 모델(DDPM)로 손글씨 숫자 생성하기

**30일 딥러닝 완전 정복 · Day 27** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- 작은 U-Net 노이즈 예측기를 MNIST로 학습한다 (T4에서 약 15분)
- 노이즈에서 숫자가 나타나는 역방향 과정을 시각화한다
- 클래스 조건 + classifier-free guidance로 원하는 숫자를 생성한다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)
tf = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x * 2 - 1)])   # [-1, 1]
dl = DataLoader(datasets.MNIST("data", train=True, download=True, transform=tf), 128, shuffle=True, num_workers=2, drop_last=True)

T = 400
betas = torch.linspace(1e-4, 0.02 * 1000 / T, T, device=device)
alphas = 1 - betas; abar = torch.cumprod(alphas, 0)
def q_sample(x0, t, eps): return abar[t].sqrt().view(-1, 1, 1, 1) * x0 + (1 - abar[t]).sqrt().view(-1, 1, 1, 1) * eps

x0, _ = next(iter(dl)); x0 = x0[:1].to(device)
fig, axs = plt.subplots(1, 8, figsize=(12, 1.8))
for a, t in zip(axs, [0, 20, 50, 100, 150, 200, 300, 399]):
    a.imshow(q_sample(x0, torch.tensor([t], device=device), torch.randn_like(x0))[0, 0].cpu(), cmap="gray"); a.set_title(f"t={t}"); a.axis("off")
plt.suptitle("forward process"); plt.show()

## 1. 작은 U-Net (시점 + 클래스 조건)
클래스 레이블 10은 '조건 없음'을 뜻합니다 (classifier-free guidance용).

In [ ]:
def t_embedding(t, dim):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    a = t.float()[:, None] * f[None]
    return torch.cat([a.sin(), a.cos()], -1)

class ResBlock(nn.Module):
    def __init__(self, c_in, c_out, d_emb):
        super().__init__()
        self.n1 = nn.GroupNorm(8, c_in); self.c1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.emb = nn.Linear(d_emb, c_out)
        self.n2 = nn.GroupNorm(8, c_out); self.c2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.skip = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.emb(e)[:, :, None, None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)

class UNet(nn.Module):
    def __init__(self, ch=64, d_emb=128, n_classes=10):
        super().__init__()
        self.d_emb = d_emb
        self.t_mlp = nn.Sequential(nn.Linear(d_emb, d_emb), nn.SiLU(), nn.Linear(d_emb, d_emb))
        self.cls = nn.Embedding(n_classes + 1, d_emb)                     # +1: 조건 없음
        self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, ch, d_emb); self.down1 = nn.Conv2d(ch, ch, 4, 2, 1)            # 28 → 14
        self.d2 = ResBlock(ch, 2 * ch, d_emb); self.down2 = nn.Conv2d(2 * ch, 2 * ch, 4, 2, 1)  # 14 → 7
        self.mid = ResBlock(2 * ch, 2 * ch, d_emb)
        self.up2 = nn.ConvTranspose2d(2 * ch, 2 * ch, 4, 2, 1); self.u2 = ResBlock(4 * ch, ch, d_emb)   # 7 → 14
        self.up1 = nn.ConvTranspose2d(ch, ch, 4, 2, 1); self.u1 = ResBlock(2 * ch, ch, d_emb)            # 14 → 28
        self.out = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, 1, 3, padding=1))
    def forward(self, x, t, y):
        e = self.t_mlp(t_embedding(t, self.d_emb)) + self.cls(y)
        h0 = self.inp(x)
        h1 = self.d1(h0, e)
        h2 = self.d2(self.down1(h1), e)
        m = self.mid(self.down2(h2), e)
        u = self.u2(torch.cat([self.up2(m), h2], 1), e)                # 스킵 연결
        u = self.u1(torch.cat([self.up1(u), h1], 1), e)
        return self.out(u)

net = UNet().to(device)
print(f"{sum(p.numel() for p in net.parameters())/1e6:.2f}M params", net(torch.randn(2, 1, 28, 28, device=device), torch.tensor([1, 2], device=device), torch.tensor([3, 10], device=device)).shape)

## 2. 학습: 노이즈 예측 MSE
10% 확률로 레이블을 '조건 없음(10)'으로 바꿔 무조건부 예측도 함께 학습합니다.

In [ ]:
opt = torch.optim.AdamW(net.parameters(), lr=2e-4)
ema = torch.optim.swa_utils.AveragedModel(net, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(0.999))   # 가중치 EMA (Day 12)
for epoch in range(15):
    for i, (x, y) in enumerate(dl):
        x, y = x.to(device), y.to(device)
        y = torch.where(torch.rand_like(y, dtype=torch.float) < 0.1, torch.full_like(y, 10), y)
        t = torch.randint(0, T, (x.size(0),), device=device); eps = torch.randn_like(x)
        loss = F.mse_loss(net(q_sample(x, t, eps), t, y), eps)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); ema.update_parameters(net)
    print(f"epoch {epoch+1}  loss {loss.item():.4f}")

## 3. 샘플링 (+ guidance)

In [ ]:
@torch.no_grad()
def sample(model, labels, w=3.0, snapshots=()):
    model.eval(); n = len(labels); x = torch.randn(n, 1, 28, 28, device=device); snaps = []
    y = torch.tensor(labels, device=device); y_null = torch.full_like(y, 10)
    for t in reversed(range(T)):
        tt = torch.full((n,), t, device=device)
        e_c, e_u = model(x, tt, y), model(x, tt, y_null)
        eps = e_u + w * (e_c - e_u)                               # classifier-free guidance
        x = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0: x = x + betas[t].sqrt() * torch.randn_like(x)
        if t in snapshots: snaps.append(x.clamp(-1, 1).cpu())
    return x.clamp(-1, 1).cpu(), snaps

gen = ema.module
imgs, snaps = sample(gen, list(range(10)) * 2, w=3.0, snapshots=(399, 300, 200, 100, 50, 0))
fig, axs = plt.subplots(len(snaps), 10, figsize=(12, 1.3 * len(snaps)))
for r, s in enumerate(snaps):
    for c in range(10): axs[r, c].imshow(s[c, 0], cmap="gray"); axs[r, c].axis("off")
plt.suptitle("reverse process (rows: t = 399 → 0)"); plt.show()

In [ ]:
for w in [0.0, 2.0, 5.0]:
    imgs, _ = sample(gen, [7] * 10, w=w)
    fig, axs = plt.subplots(1, 10, figsize=(12, 1.5))
    for a, im in zip(axs, imgs): a.imshow(im[0], cmap="gray"); a.axis("off")
    plt.suptitle(f"label 7, guidance w = {w}"); plt.show()

## 과제
- guidance w가 커질수록 '7다움'과 다양성이 어떻게 변하나요?
- `T=100`으로 줄여 다시 학습하면 품질과 속도는?